# Word attention in Stable Diffusion

Explore where words receive spatial attention, then compare their heatmaps.

**Two stages:** an offline numerical sanity check, followed by optional real Stable Diffusion generation with DAAM. The offline arrays are synthetic, not model results. See the README for installation and validation status.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "diffusion_attention").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the cloned repository or its notebooks folder.")
sys.path.insert(0, str(ROOT))
from diffusion_attention.analysis import continuous_iou, binary_mask


## 1. Check the metric without a model

These small arrays give an independently checkable score of 1/3. Uniform maps have no localized region, so their masks are empty.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a = np.array([[1., 1.], [0., 0.]])
b = np.array([[0., 1.], [0., 1.]])
print(f"Synthetic example IoU: {continuous_iou(a, b):.6f}")
assert np.isclose(continuous_iou(a, b), 1 / 3)
assert not binary_mask(np.zeros((2, 2))).any()
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for axis, array, title in zip(axes, [a, b], ["Synthetic map A", "Synthetic map B"]):
    axis.imshow(array, cmap="magma", vmin=0, vmax=1)
    axis.set_title(title)
    axis.axis("off")
plt.show()


## 2. Inspect a historical model result

This figure was extracted from the original uploaded `Daam_run_Model.ipynb`. It is not output from the new code. Prompt: **An angry, bald man writing papers**. Model recorded in source: `stabilityai/stable-diffusion-2-base`. Seed and exact runtime versions are unavailable.


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / "docs/assets/historical_attention.png")))


## 3. Run a new generation (optional)

Use a Python 3.10/3.11 kernel with `requirements-model.lock.txt` installed as described in the README. Authentication, if needed, belongs in `huggingface-cli login` or `HF_TOKEN`, never in a saved code cell. CUDA is recommended; CPU uses float32 and is slow. Set `RUN_GENERATION = True` to download/load the model and run. Choose a fresh output folder for every run.


In [ ]:
RUN_GENERATION = False
PROMPT = "A red fire truck parked on a quiet street"
EXPRESSION = "fire truck"
SEED = 42
STEPS = 20
DEVICE = "auto"
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
REVISION = None  # Set a model commit SHA to pin the weights.
OUTPUT = ROOT / "outputs/notebook-demo"


In [ ]:
if RUN_GENERATION:
    from diffusion_attention.demo import load_pipeline, run_example
    pipe, device = load_pipeline(MODEL_ID, DEVICE, REVISION)
    scores = run_example(pipe, PROMPT, EXPRESSION, OUTPUT, seed=SEED, steps=STEPS,
                         model_id=MODEL_ID, revision=REVISION, device=device)
    display(Image(filename=str(OUTPUT / "attention.png")))
    print(scores["pairwise_scores"])
    print("Missing words:", scores["missing_words"])
else:
    print("Model generation skipped. Set RUN_GENERATION = True after installing the model environment.")


## Interpreting the result

Continuous IoU compares raw heatmap amplitudes: `sum(min(A,B)) / sum(max(A,B))`. This differs from binary-mask IoU and can change with relative attention intensity. Display overlays are normalized separately; scores are not. Repeated occurrences of a word are aggregated by DAAM. Missing maps are reported, never replaced with fake zero-valued scores.

A high score means overlapping attention, **not** proof of correct linguistic understanding or compositionality. Compare multiple prompts and seeds before drawing conclusions.
